In [1]:
import pandas as pd

df = pd.read_csv('clean_data_after_eda.csv')
df["date_activ"] = pd.to_datetime(df["date_activ"], format='%Y-%m-%d')
df["date_end"] = pd.to_datetime(df["date_end"], format='%Y-%m-%d')
df["date_modif_prod"] = pd.to_datetime(df["date_modif_prod"], format='%Y-%m-%d')
df["date_renewal"] = pd.to_datetime(df["date_renewal"], format='%Y-%m-%d')
print(df.shape)

(14606, 44)


In [2]:
price_df = pd.read_csv('price_data.csv')
price_df["price_date"] = pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')

monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({'price_off_peak_var': 'mean', 'price_off_peak_fix': 'mean'}).reset_index()

jan_prices = monthly_price_by_id.groupby('id').first().reset_index()
dec_prices = monthly_price_by_id.groupby('id').last().reset_index()

diff = pd.merge(dec_prices.rename(columns={'price_off_peak_var': 'dec_1', 'price_off_peak_fix': 'dec_2'}), jan_prices.drop(columns='price_date'), on='id')
diff['offpeak_diff_dec_january_energy'] = diff['dec_1'] - diff['price_off_peak_var']
diff['offpeak_diff_dec_january_power'] = diff['dec_2'] - diff['price_off_peak_fix']
diff = diff[['id', 'offpeak_diff_dec_january_energy','offpeak_diff_dec_january_power']]
print(diff.head())

                                 id  offpeak_diff_dec_january_energy  \
0  0002203ffbb812588b632b9e628cc38d                        -0.006192   
1  0004351ebdd665e6ee664792efc4fd13                        -0.004104   
2  0010bcc39e42b3c2131ed2ce55246e3c                         0.050443   
3  0010ee3855fdea87602a5b7aba8e42de                        -0.010018   
4  00114d74e963e47177db89bc70108537                        -0.003994   

   offpeak_diff_dec_january_power  
0                        0.162916  
1                        0.177779  
2                        1.500000  
3                        0.162916  
4                       -0.000001  


In [3]:
df = pd.merge(df, diff, on='id', how='left')
print("After merging price diff:", df.shape)

After merging price diff: (14606, 46)


In [4]:
# Extract useful info from date columns
df['activation_month'] = df['date_activ'].dt.month
df['activation_year'] = df['date_activ'].dt.year
df['contract_end_month'] = df['date_end'].dt.month
df['tenure_years'] = (df['date_end'] - df['date_activ']).dt.days / 365
print(df[['activation_month', 'activation_year', 'contract_end_month', 'tenure_years']].head())

   activation_month  activation_year  contract_end_month  tenure_years
0                 6             2013                   6      3.002740
1                 8             2009                   8      7.030137
2                 4             2010                   4      6.005479
3                 3             2010                   3      6.005479
4                 1             2010                   3      6.150685


In [5]:
# Drop columns that are not useful for prediction
cols_to_drop = ['date_activ', 'date_end', 'date_modif_prod', 'date_renewal']
df = df.drop(columns=cols_to_drop)
print("After dropping date columns:", df.shape)

After dropping date columns: (14606, 46)


In [6]:
df.to_csv('data_after_feature_engineering.csv', index=False)
print("Done! File saved.")
print(df.head(3))

Done! File saved.
                                 id                     channel_sales  \
0  24011ae4ebbe3035111d65fa7c15bc57  foosdfpfkusacimwkcsosbicdxkicaua   
1  d29c2c54acc38ff3c0614d0a653813dd                           MISSING   
2  764c75f661154dac3a6c254cd082ea7d  foosdfpfkusacimwkcsosbicdxkicaua   

   cons_12m  cons_gas_12m  cons_last_month  forecast_cons_12m  \
0         0         54946                0               0.00   
1      4660             0                0             189.95   
2       544             0                0              47.96   

   forecast_cons_year  forecast_discount_energy  forecast_meter_rent_12m  \
0                   0                       0.0                     1.78   
1                   0                       0.0                    16.27   
2                   0                       0.0                    38.72   

   forecast_price_energy_off_peak  ...  var_6m_price_off_peak  \
0                        0.114481  ...               2.086